## Init

In [0]:
from delta.tables import DeltaTable
import json
import time
import requests
from datetime import datetime, date

## Read the cities table

In [0]:
df_cities = (
    spark.read
    .table("weather.cities")
    .select("state_code", "lat", "lon")
    .orderBy("state_code")
)

# Convert rows to dictionaries
cities_config = [row.asDict() for row in df_cities.collect()]

## Read from Open-Meteo API 

In [0]:
raw_records = []

def extract_open_meteo_with_retry(lat, lon, date_str, max_retries=3):
    url = (
        f"https://api.open-meteo.com/v1/forecast?"
        f"latitude={lat}&longitude={lon}"
        f"&hourly=temperature_2m,relative_humidity_2m,precipitation"
        f"&start_date={date_str}&end_date={date_str}"
        f"&timezone=auto"
    )
    
    for attempt in range(1, max_retries + 1):
        response = requests.get(url)
        data = response.json()
        times = data.get("hourly", {}).get("time", [])

        if len(times) == 24:
            print(f" Attempt {attempt}: Full ingestion (24/24 hours).")
            return url, data, "SUCCESS"

        print(f" Attempt {attempt}: Incomplete ingestion ({len(times)}/24 hours). Retrying...")
        time.sleep(5)

    print(" Alert: Maximum retries reached. Saving incomplete payload...")
    return url, data, "PARTIAL_SUCCESS"

In [ ]:
# Iterate over the cities and query Open-Meteo
for c in cities_config:
    url, data, status = extract_open_meteo_with_retry(c["lat"], c["lon"], date.today().isoformat())

    raw_records.append({
        "state_code": c["state_code"],
        "raw_payload": json.dumps(data),
        "ingested_at": datetime.now().isoformat(),
        "source_endpoint": url
    })

# Convert to DataFrame from PySpark
df = spark.createDataFrame(raw_records)

In [0]:
# Order and cast columns
df_bronze = (
    df
    .select(
        col("state_code").cast("int"),
        col("raw_payload"),
        col("ingested_at").cast("timestamp"),
        col("source_endpoint")
    )
)

## Write data in bronze table
Using MERGE with (state_code + date of ingestion) as unique key 

In [0]:
table_name = "weather.raw_data"

if not spark.catalog.tableExists(table_name):
    (
        df_bronze
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(table_name)
    )
else: 
    target_table = DeltaTable.forName(spark, table_name)

    (
        target_table.alias("target")
        .merge(
            df_bronze.alias("source"), 
            condition="""
            target.state_code = source.state_code AND 
            to_date(target.ingested_at) = to_date(source.ingested_at)
            """
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

## Sanity check

In [0]:
%sql
select * 
from weather.raw_data
order by state_code, ingested_at desc
limit 10